# COMP3710 Part 4 Task 2: OASIS multi-class U-Net

This notebook implements the required OASIS brain-MRI segmentation pipeline in PyTorch. It is staged so the cheap architecture check can run independently from the real-data smoke test, full training, and final test evaluation.

The model predicts four mutually exclusive classes. Masks are decoded from pixel values `0, 85, 170, 255` into labels `0, 1, 2, 3`, then one-hot encoded for the categorical loss. The final test report prints discrete DSC for all four classes separately.

In [ ]:
import json
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset

SEED = int(os.environ.get('UNET_SEED', '42'))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_ROOT = Path(os.environ.get('UNET_DATA_ROOT', '/home/groups/comp3710/OASIS'))
IMAGE_SIZE = int(os.environ.get('UNET_IMG', '128'))
EPOCHS = int(os.environ.get('UNET_EPOCHS', '30'))
BATCH_SIZE = int(os.environ.get('UNET_BATCH', '8'))
NUM_WORKERS = int(os.environ.get('UNET_WORKERS', '2'))
STAGE = os.environ.get('UNET_STAGE', 'full').lower()
OUTPUT_DIR = Path(os.environ.get('UNET_OUTPUT', 'unet_outputs'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print({'stage': STAGE, 'data_root': str(DATA_ROOT), 'image_size': IMAGE_SIZE, 'epochs': EPOCHS, 'device': str(DEVICE)})

## 1. Dataset pairing and preprocessing

Every `case_*.png` must have exactly one matching `seg_*.png`. Images use bilinear resizing; masks use nearest-neighbor resizing so class labels are never interpolated into invalid values.

In [ ]:
class OasisSliceDataset(Dataset):
    def __init__(self, root, split, image_size=128):
        self.root = Path(root)
        self.split = split
        self.image_size = int(image_size)
        self.image_dir = self.root / f'keras_png_slices_{split}'
        self.mask_dir = self.root / f'keras_png_slices_seg_{split}'
        self.image_paths = sorted(self.image_dir.glob('case_*.png'))
        if not self.image_paths:
            raise FileNotFoundError(f'No case_*.png files in {self.image_dir}')
        self.mask_paths = []
        for image_path in self.image_paths:
            mask_path = self.mask_dir / image_path.name.replace('case_', 'seg_', 1)
            if not mask_path.exists():
                raise FileNotFoundError(f'Missing mask for {image_path.name}: {mask_path}')
            self.mask_paths.append(mask_path)
        assert len(self.image_paths) == len(self.mask_paths)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image = Image.open(self.image_paths[index]).convert('L')
        mask = Image.open(self.mask_paths[index]).convert('L')
        image = image.resize((self.image_size, self.image_size), Image.Resampling.BILINEAR)
        mask = mask.resize((self.image_size, self.image_size), Image.Resampling.NEAREST)
        image_array = np.asarray(image, dtype=np.float32) / 255.0
        # The four allowed grayscale mask values are 0, 85, 170, and 255.
        mask_array = np.asarray(mask, dtype=np.int64)
        labels = np.rint(mask_array / 85.0).astype(np.int64)
        if labels.min() < 0 or labels.max() > 3:
            raise ValueError(f'Invalid mask labels in {self.mask_paths[index]}: {np.unique(labels)}')
        return torch.from_numpy(image_array[None]), torch.from_numpy(labels)

def make_loader(dataset, batch_size, shuffle, drop_last=False):
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'), persistent_workers=(NUM_WORKERS > 0)
    )

train_dataset = OasisSliceDataset(DATA_ROOT, 'train', IMAGE_SIZE)
validate_dataset = OasisSliceDataset(DATA_ROOT, 'validate', IMAGE_SIZE)
test_dataset = OasisSliceDataset(DATA_ROOT, 'test', IMAGE_SIZE)
print('split sizes:', len(train_dataset), len(validate_dataset), len(test_dataset))
print('first pair:', train_dataset.image_paths[0].name, train_dataset.mask_paths[0].name)

## 2. U-Net architecture

The encoder reduces resolution to learn context. The decoder restores resolution. Skip connections concatenate high-resolution encoder features into the matching decoder level. Without them, repeated downsampling discards boundary detail; the skips restore that spatial information before the final pixel classifier.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class UNet(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.enc1 = DoubleConv(1, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = DoubleConv(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = DoubleConv(64, 32)
        # Four logits per pixel; softmax is applied only for probabilities.
        self.head = nn.Conv2d(32, num_classes, 1)

    def forward(self, x):
        skip1 = self.enc1(x)
        skip2 = self.enc2(self.pool(skip1))
        bottleneck = self.enc3(self.pool(skip2))
        up2 = self.up2(bottleneck)
        up2 = torch.cat([up2, skip2], dim=1)
        dec2 = self.dec2(up2)
        up1 = self.up1(dec2)
        up1 = torch.cat([up1, skip1], dim=1)
        dec1 = self.dec1(up1)
        return self.head(dec1)

model = UNet(num_classes=4).to(DEVICE)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f'parameters: {parameter_count:,}')

In [ ]:
# Stage 1: architecture-only sanity check with random data.
if STAGE in {'architecture', 'smoke', 'full', 'test'}:
    model.train()
    random_images = torch.rand(2, 1, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
    random_labels = torch.randint(0, 4, (2, IMAGE_SIZE, IMAGE_SIZE), device=DEVICE)
    random_one_hot = F.one_hot(random_labels, num_classes=4).permute(0, 3, 1, 2).float()
    random_logits = model(random_images)
    print('random output:', tuple(random_logits.shape))
    assert random_logits.shape == (2, 4, IMAGE_SIZE, IMAGE_SIZE)
    architecture_loss = F.cross_entropy(random_logits, random_labels)
    architecture_loss.backward()
    missing_gradients = [name for name, p in model.named_parameters() if p.grad is None]
    assert not missing_gradients, missing_gradients
    model.zero_grad(set_to_none=True)
    print('architecture check passed; one-hot shape:', tuple(random_one_hot.shape))

## 3. Imbalance-aware categorical loss

Plain cross-entropy can achieve a deceptively good pixel accuracy by favoring the 72.25% background class. The loss below combines weighted categorical cross-entropy, soft foreground Dice, and a Tversky term that penalizes missed foreground pixels. The final metric is still hard, discrete DSC on the untouched test set.

In [ ]:
# These weights reflect the supplied class distribution. Background is downweighted.
CLASS_WEIGHTS = torch.tensor([0.25, 1.0, 0.8, 0.8], device=DEVICE)

def soft_foreground_dice_loss(logits, one_hot, smooth=1e-6):
    probabilities = torch.softmax(logits, dim=1)
    intersection = (probabilities * one_hot).sum(dim=(0, 2, 3))
    denominator = probabilities.sum(dim=(0, 2, 3)) + one_hot.sum(dim=(0, 2, 3))
    dice = (2 * intersection + smooth) / (denominator + smooth)
    return 1.0 - dice[1:].mean()

def tversky_loss(logits, one_hot, alpha=0.3, beta=0.7, smooth=1e-6):
    probabilities = torch.softmax(logits, dim=1)
    foreground_probabilities = probabilities[:, 1:]
    foreground_truth = one_hot[:, 1:]
    true_positive = (foreground_probabilities * foreground_truth).sum(dim=(0, 2, 3))
    false_positive = (foreground_probabilities * (1 - foreground_truth)).sum(dim=(0, 2, 3))
    false_negative = ((1 - foreground_probabilities) * foreground_truth).sum(dim=(0, 2, 3))
    score = (true_positive + smooth) / (true_positive + alpha * false_positive + beta * false_negative + smooth)
    return 1.0 - score.mean()

def segmentation_loss(logits, labels):
    one_hot = F.one_hot(labels, num_classes=4).permute(0, 3, 1, 2).float()
    cross_entropy = F.cross_entropy(logits, labels, weight=CLASS_WEIGHTS)
    return 0.5 * cross_entropy + soft_foreground_dice_loss(logits, one_hot) + tversky_loss(logits, one_hot)

@torch.no_grad()
def per_class_dice_from_predictions(predictions, labels, num_classes=4):
    scores = []
    for class_index in range(num_classes):
        truth = labels == class_index
        predicted = predictions == class_index
        denominator = truth.sum().item() + predicted.sum().item()
        intersection = (truth & predicted).sum().item()
        scores.append(1.0 if denominator == 0 else 2.0 * intersection / denominator)
    return np.asarray(scores, dtype=np.float64)

## 4. Stage 2 smoke test

This uses only a few hundred real pairs and two epochs. It checks that pairing, resizing, one-hot targets, loss, backpropagation, and visualization all work before spending cluster time on the full run.

In [ ]:
if STAGE == 'smoke':
    smoke_count = min(256, len(train_dataset))
    smoke_dataset = Subset(train_dataset, range(smoke_count))
    smoke_loader = make_loader(smoke_dataset, min(BATCH_SIZE, 8), shuffle=True)
    smoke_model = UNet(4).to(DEVICE)
    smoke_optimizer = torch.optim.Adam(smoke_model.parameters(), lr=3e-4)
    for smoke_epoch in range(2):
        smoke_model.train()
        losses = []
        for images, labels in smoke_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            smoke_optimizer.zero_grad(set_to_none=True)
            loss = segmentation_loss(smoke_model(images), labels)
            assert torch.isfinite(loss), loss
            loss.backward()
            smoke_optimizer.step()
            losses.append(loss.item())
        print(f'smoke epoch {smoke_epoch + 1}: loss={np.mean(losses):.4f}')
    smoke_model.eval()
    with torch.no_grad():
        smoke_images, smoke_labels = next(iter(smoke_loader))
        smoke_predictions = smoke_model(smoke_images.to(DEVICE)).argmax(1).cpu().numpy()
    figure, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(smoke_images[0, 0], cmap='gray')
    axes[1].imshow(smoke_labels[0], cmap='viridis', vmin=0, vmax=3)
    axes[2].imshow(smoke_predictions[0], cmap='viridis', vmin=0, vmax=3)
    for axis, title in zip(axes, ['MR image', 'ground truth', 'prediction']):
        axis.set_title(title); axis.axis('off')
    figure.tight_layout(); figure.savefig(OUTPUT_DIR / 'smoke_triptych.png', dpi=160); plt.show()

## 5. Full training

The full stage trains on all training pairs, monitors validation DSC each epoch, and saves the best model by mean foreground validation DSC. The test set is not used for model selection.

In [ ]:
def evaluate_loader(model, loader):
    model.eval()
    all_scores = []
    with torch.no_grad():
        for images, labels in loader:
            logits = model(images.to(DEVICE))
            predictions = logits.argmax(1).cpu()
            all_scores.append(per_class_dice_from_predictions(predictions, labels))
    return np.mean(np.stack(all_scores), axis=0)


def train_full_model():
    train_loader = make_loader(train_dataset, BATCH_SIZE, shuffle=True)
    validate_loader = make_loader(validate_dataset, BATCH_SIZE, shuffle=False)
    trained_model = UNet(4).to(DEVICE)
    optimizer = torch.optim.AdamW(trained_model.parameters(), lr=3e-4, weight_decay=1e-5)
    best_score = -1.0
    best_path = OUTPUT_DIR / 'best_unet.pt'
    history = {'train_loss': [], 'val_dice': []}
    for epoch in range(EPOCHS):
        trained_model.train()
        epoch_losses = []
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = segmentation_loss(trained_model(images), labels)
            if not torch.isfinite(loss):
                raise FloatingPointError(f'Non-finite loss at epoch {epoch + 1}')
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trained_model.parameters(), 5.0)
            optimizer.step()
            epoch_losses.append(loss.item())
        val_scores = evaluate_loader(trained_model, validate_loader)
        mean_foreground = float(val_scores[1:].mean())
        history['train_loss'].append(float(np.mean(epoch_losses)))
        history['val_dice'].append(val_scores.tolist())
        train_loss_value = history['train_loss'][-1]
        print(f'Epoch {epoch + 1}/{EPOCHS}: loss={train_loss_value:.4f} val_dice={val_scores}')
        if mean_foreground > best_score:
            best_score = mean_foreground
            torch.save({'model': trained_model.state_dict(), 'epoch': epoch + 1, 'val_dice': val_scores.tolist()}, best_path)
    with open(OUTPUT_DIR / 'training_results.json', 'w', encoding='utf-8') as handle:
        json.dump(history, handle, indent=2)
    return trained_model, best_path, history

trained_model = None
best_checkpoint = OUTPUT_DIR / 'best_unet.pt'
history = None
if STAGE == 'full':
    trained_model, best_checkpoint, history = train_full_model()
    print('best checkpoint:', best_checkpoint)

## 6. Held-out test evaluation and visual evidence

This cell loads the best validation checkpoint, computes DSC for classes 0, 1, 2, and 3 separately on the held-out test set, saves JSON/PNG evidence, and creates side-by-side qualitative examples plus a per-class DSC bar chart.

In [ ]:
def save_test_evidence():
    checkpoint = torch.load(best_checkpoint, map_location=DEVICE)
    final_model = UNet(4).to(DEVICE)
    final_model.load_state_dict(checkpoint['model'])
    final_model.eval()
    test_loader = make_loader(test_dataset, BATCH_SIZE, shuffle=False)
    score_batches = []
    preview_images, preview_labels, preview_predictions = [], [], []
    with torch.no_grad():
        for images, labels in test_loader:
            predictions = final_model(images.to(DEVICE)).argmax(1).cpu()
            score_batches.append(per_class_dice_from_predictions(predictions, labels))
            if len(preview_images) < 4:
                remaining = 4 - len(preview_images)
                preview_images.extend(images[:remaining, 0].numpy())
                preview_labels.extend(labels[:remaining].numpy())
                preview_predictions.extend(predictions[:remaining].numpy())
    scores = np.mean(np.stack(score_batches), axis=0)
    result = {'checkpoint': str(best_checkpoint), 'test_dsc_per_class': scores.tolist(), 'required_foreground_threshold': 0.9}
    with open(OUTPUT_DIR / 'test_results.json', 'w', encoding='utf-8') as handle:
        json.dump(result, handle, indent=2)
    print('Discrete test DSC per class:')
    for class_index, score in enumerate(scores):
        print(f'  class {class_index}: {score:.4f}')
    figure, axes = plt.subplots(4, 3, figsize=(12, 14))
    for row in range(4):
        axes[row, 0].imshow(preview_images[row], cmap='gray'); axes[row, 0].set_title('MR image')
        axes[row, 1].imshow(preview_labels[row], cmap='viridis', vmin=0, vmax=3); axes[row, 1].set_title('ground truth')
        axes[row, 2].imshow(preview_predictions[row], cmap='viridis', vmin=0, vmax=3); axes[row, 2].set_title('prediction')
        for column in range(3): axes[row, column].axis('off')
    figure.tight_layout(); figure.savefig(OUTPUT_DIR / 'test_triptychs.png', dpi=160); plt.show()
    bar = plt.figure(figsize=(7, 4))
    plt.bar([str(index) for index in range(4)], scores)
    plt.axhline(0.9, color='red', linestyle='--', label='required foreground threshold')
    plt.ylim(0, 1); plt.xlabel('class'); plt.ylabel('discrete DSC'); plt.legend(); plt.tight_layout()
    bar.savefig(OUTPUT_DIR / 'test_dsc_per_class.png', dpi=160); plt.show()
    torch.save(final_model.state_dict(), OUTPUT_DIR / 'unet_final.pt')

if STAGE in {'test', 'full'} and best_checkpoint.exists():
    save_test_evidence()
else:
    print('Test stage skipped; run with UNET_STAGE=full or test after a checkpoint exists.')

## 7. Demonstrator Q&A

**What do skip connections do?** They preserve high-resolution encoder features and concatenate them into the decoder. Without them, downsampling loses boundary detail.

**Why use Dice rather than only cross-entropy?** Background occupies most pixels, so cross-entropy can look good while a small tissue class is missed. Dice directly measures overlap and foreground Dice gives those classes a stronger training signal.

**What does DSC measure, and how is it different from pixel accuracy?** DSC measures overlap for one class: $2|A\cap B|/(|A|+|B|)$. Pixel accuracy counts all correct pixels and can be dominated by background.

**Why one-hot output?** Four softmax channels represent the four mutually exclusive labels. One-hot targets make the categorical output contract explicit and satisfy the lab requirement.

**Why is the smallest class hardest?** It has fewer pixels and less gradient signal. A model can improve the dominant background while neglecting the rare class, so class-aware Dice/Tversky terms are useful.

**What does the current result prove?** It proves the pipeline, inference, visualizations, and per-class evaluation work. It does not prove full marks unless every foreground test DSC exceeds 0.9.